# 15 — LLM-as-a-Judge and Human Evaluation

## Scenario
Northstar generates automated email responses to customers. 
Unlike classification tasks (where there is one "correct" answer like `Refund`), emails are open-ended and generative. We cannot evaluate them using simple `actual == expected` string matching.

**The Problem:** We need a way to programmatically evaluate the *quality* of generated text at scale.

In [ ]:
import os
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

# 1. The Generative Task
customer_message = "My coffee mug arrived shattered. I'm furious!"
agent_prompt = f"Write a response to this customer: {customer_message}"

response_agent = client.models.generate_content(model=MODEL_ID, contents=agent_prompt)
candidate_email = response_agent.text

print("--- CANDIDATE EMAIL TO BE EVALUATED ---")
print(candidate_email)


## Step 1: Defining the Rubric and Structured Output

To automate the evaluation, we employ a second LLM prompt (the "Judge"). We use Pydantic to ensure the Judge returns a structured score. 

**Pro Tip:** Always ask the model to provide its `reasoning` *before* it outputs its `score`. LLMs generate tokens sequentially; forcing it to explain itself first usually results in a much more accurate final score (Chain of Thought).

In [ ]:
class EvaluationResult(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning evaluating the email against the rubric.")
    score: int = Field(description="Integer score from 1 to 5.")

rubric = """\nYou are an expert customer service manager evaluating agent emails.\nEvaluate the provided candidate email against the following rubric:\n\n1 - Terrible: Argumentative, defensive, or completely ignores the customer's problem.\n3 - Acceptable: Addresses the problem but lacks empathy or reads like a robot.\n5 - Excellent: Deeply empathetic, immediately apologizes, and offers a clear resolution.\n"""

judge_prompt = f"""\n{rubric}\n\n--- CANDIDATE EMAIL ---\n{candidate_email}\n"""

response_judge = client.models.generate_content(
    model=MODEL_ID,
    contents=judge_prompt,
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=EvaluationResult,
    )
)

print("\n--- JUDGE EVALUATION ---")
eval_result = EvaluationResult.model_validate_json(response_judge.text)
print(f"Score: {eval_result.score}/5")
print(f"Reasoning: {eval_result.reasoning}")


## Conclusion: The Need for Human Baselines

While LLM-as-a-Judge is the industry standard for scaling evaluations, judges are notoriously flawed:
1. **Verbosity Bias:** LLMs tend to give higher scores to longer answers, regardless of quality.
2. **Self-Preference:** An LLM will generally score its own outputs higher than those generated by a rival model.
3. **Position Bias (in A/B testing):** If asked to choose between Candidate A and Candidate B, the model is often biased toward whichever is presented first.

Because of this, an LLM Judge should never be treated as Ground Truth. You must periodically calculate the **Agreement Rate** between your LLM Judge and Human Experts to ensure the Judge's calibration hasn't drifted.